In [1]:
!pip install -q numpy pandas scikit-learn matplotlib

In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
import os

In [4]:
LOCAL_DIR = "/content/logsentinel_data"
RAW_DIR = os.path.join(LOCAL_DIR, "raw")
OUT_DIR = os.path.join(LOCAL_DIR, "processed")

In [5]:
DRIVE_ROOT = "/content/drive/MyDrive/Aegis-CyberSec-Guard"
DRIVE_OUT = os.path.join(DRIVE_ROOT, "logsentinel", "dataV1")


In [6]:
HDFS_ZIP_URL = "https://zenodo.org/records/8196385/files/HDFS_v1.zip?download=1"


In [7]:
MAX_SEQ_LEN = 40           # sequences longer than this are truncated
PAD_VALUE = 0              # 0 is reserved for padding; real event IDs start at 1


In [8]:

TEST_SIZE = 0.15
VAL_SIZE = 0.15            # of the remaining after test split
SEED = 3407

In [9]:

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)


In [10]:
print(f"Local work dir: {LOCAL_DIR}")
print(f"Final tensors will be copied to: {DRIVE_OUT}")


Local work dir: /content/logsentinel_data
Final tensors will be copied to: /content/drive/MyDrive/Aegis-CyberSec-Guard/logsentinel/dataV1


In [11]:

import zipfile
import urllib.request


In [12]:
zip_path = os.path.join(RAW_DIR, "HDFS_v1.zip")


In [13]:
if not os.path.exists(zip_path):
    print("Downloading HDFS_v1.zip from Zenodo (this is a large file)...")
    urllib.request.urlretrieve(HDFS_ZIP_URL, zip_path)
    print(f"Downloaded: {os.path.getsize(zip_path)/1e6:.1f} MB")
else:
    print("Zip already present, skipping download.")


Downloaded: 186.6 MB


In [14]:
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(RAW_DIR)
    print("Extracted files:")
    for name in z.namelist():
        print(f"  {name}")


Extracted files:
  HDFS.log
  preprocessed/
  preprocessed/anomaly_label.csv
  preprocessed/Event_occurrence_matrix.csv
  preprocessed/Event_traces.csv
  preprocessed/HDFS.log_templates.csv
  preprocessed/HDFS.npz
  README.md


In [15]:

import glob


In [16]:
def find_file(root, filename):
    matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
    return matches[0] if matches else None


In [17]:
event_traces_path = find_file(RAW_DIR, "Event_traces.csv")
anomaly_label_path = find_file(RAW_DIR, "anomaly_label.csv")
templates_path = find_file(RAW_DIR, "HDFS_templates.csv")


In [18]:
print(f"Event_traces.csv: {event_traces_path}")
print(f"anomaly_label.csv: {anomaly_label_path}")
print(f"HDFS_templates.csv: {templates_path}")


Event_traces.csv: /content/logsentinel_data/raw/preprocessed/Event_traces.csv
anomaly_label.csv: /content/logsentinel_data/raw/preprocessed/anomaly_label.csv
HDFS_templates.csv: None


In [19]:
assert event_traces_path, "Event_traces.csv not found — inspect the extracted structure"


In [20]:
import pandas as pd


In [21]:
import ast

In [22]:

traces = pd.read_csv(event_traces_path)
print(f"Loaded {len(traces)} rows, columns: {list(traces.columns)}")


Loaded 575061 rows, columns: ['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']


In [23]:
# Resolve column names case-insensitively.
def resolve_col(df, *candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None


In [24]:
block_col = resolve_col(traces, "BlockId", "block_id", "Block")
seq_col = resolve_col(traces, "Features", "EventSequence", "Sequence", "Events")
label_col_in_traces = resolve_col(traces, "Label", "Type")



In [25]:
print(f"block column: {block_col}")
print(f"sequence column: {seq_col}")
print(f"label column in traces (if any): {label_col_in_traces}")


block column: BlockId
sequence column: Features
label column in traces (if any): Label


In [26]:
assert block_col and seq_col, "Could not resolve block/sequence columns — check Cell 6 output"


In [27]:
def parse_sequence(raw):
    """
    Turn a stored sequence into a list of event-token strings.
    HDFS_v1 stores them as "[E5,E22,E5,...,E21]" — unquoted tokens inside brackets,
    so ast.literal_eval fails and we must strip the brackets ourselves.
    """
    if not isinstance(raw, str):
        return []
    s = raw.strip()
    # Strip the surrounding brackets before splitting; otherwise the first and last
    # tokens come out as "[E5" and "E21]" and pollute the vocabulary.
    if s.startswith("["):
        s = s[1:]
    if s.endswith("]"):
        s = s[:-1]
    tokens = [t.strip() for t in s.replace(",", " ").split()]
    return [t for t in tokens if t]


In [28]:
traces["parsed_seq"] = traces[seq_col].apply(parse_sequence)


In [29]:
# Sequence-length distribution: this is what MAX_SEQ_LEN must respect.
seq_lens = traces["parsed_seq"].apply(len)
print("\nSequence length distribution:")
print(f"  min: {seq_lens.min()}")
print(f"  p50: {int(seq_lens.quantile(0.50))}")
print(f"  p95: {int(seq_lens.quantile(0.95))}")
print(f"  p99: {int(seq_lens.quantile(0.99))}")
print(f"  max: {seq_lens.max()}")
over = (seq_lens > MAX_SEQ_LEN).sum()
print(f"  over MAX_SEQ_LEN={MAX_SEQ_LEN}: {over} ({100*over/len(traces):.2f}%) will be truncated")



Sequence length distribution:
  min: 2
  p50: 19
  p95: 28
  p99: 33
  max: 298
  over MAX_SEQ_LEN=40: 710 (0.12%) will be truncated


In [30]:
# Cell 8 - Attach labels
# HDFS_v1's Event_traces.csv carries a Success/Fail column, but that is NOT the official
# ground truth. anomaly_label.csv is the labeled set used across the literature, so we
# always join against it and ignore the in-traces Label column.

assert anomaly_label_path, "anomaly_label.csv is required — it is the ground truth"

labels = pd.read_csv(anomaly_label_path)
lbl_block = resolve_col(labels, "BlockId", "block_id")
lbl_label = resolve_col(labels, "Label", "Type")
print(f"anomaly_label.csv columns -> block: {lbl_block}, label: {lbl_label}")
print(f"Raw label values: {labels[lbl_label].unique()}")

labels = labels[[lbl_block, lbl_label]].rename(
    columns={lbl_block: block_col, lbl_label: "label_str"}
)

# Drop the misleading Success/Fail column before joining to avoid a name clash.
if label_col_in_traces and label_col_in_traces in traces.columns:
    traces = traces.drop(columns=[label_col_in_traces])

before_join = len(traces)
traces = traces.merge(labels, on=block_col, how="inner")
print(f"Rows before join: {before_join} | after join: {len(traces)}")

traces["label"] = traces["label_str"].str.strip().str.lower().eq("anomaly").astype(int)

n_anom = int(traces["label"].sum())
n_total = len(traces)
print(f"\nLabel distribution:")
print(f"  Normal:  {n_total - n_anom} ({100*(n_total-n_anom)/n_total:.2f}%)")
print(f"  Anomaly: {n_anom} ({100*n_anom/n_total:.2f}%)")
assert n_anom > 0, "Zero anomalies — the join or the label mapping failed"

anomaly_label.csv columns -> block: BlockId, label: Label
Raw label values: ['Normal' 'Anomaly']
Rows before join: 575061 | after join: 575061

Label distribution:
  Normal:  558223 (97.07%)
  Anomaly: 16838 (2.93%)


In [31]:

# Cell 9 - Build the event-ID vocabulary
# Map each distinct event token (E1, E2, ...) to an integer >= 1. 0 stays reserved for pad.
# UNKNOWN gets its own ID for any token unseen at fit time (robustness at inference).

from collections import Counter

all_tokens = Counter()
for seq in traces["parsed_seq"]:
    all_tokens.update(seq)

# Deterministic ordering for reproducibility.
vocab = {tok: i + 1 for i, (tok, _) in enumerate(sorted(all_tokens.items()))}
UNKNOWN_ID = len(vocab) + 1
vocab["<UNK>"] = UNKNOWN_ID

VOCAB_SIZE = len(vocab) + 1  # +1 because index 0 (pad) is not in the dict

print(f"Distinct event types: {len(all_tokens)}")
print(f"Vocab size (incl. pad=0 and <UNK>={UNKNOWN_ID}): {VOCAB_SIZE}")
print(f"Sample mapping: {dict(list(vocab.items())[:8])}")



Distinct event types: 29
Vocab size (incl. pad=0 and <UNK>=30): 31
Sample mapping: {'E1': 1, 'E10': 2, 'E11': 3, 'E12': 4, 'E13': 5, 'E14': 6, 'E15': 7, 'E16': 8}


In [32]:

# Cell 10 - Encode and pad sequences into a fixed-shape integer matrix
import numpy as np

def encode_and_pad(seq, vocab, max_len, pad_value, unk_id):
    ids = [vocab.get(tok, unk_id) for tok in seq[:max_len]]
    if len(ids) < max_len:
        ids = ids + [pad_value] * (max_len - len(ids))
    return ids


X = np.array(
    [encode_and_pad(seq, vocab, MAX_SEQ_LEN, PAD_VALUE, UNKNOWN_ID)
     for seq in traces["parsed_seq"]],
    dtype=np.int32,
)
y = traces["label"].to_numpy(dtype=np.int32)

print(f"X shape: {X.shape}  (sessions, MAX_SEQ_LEN)")
print(f"y shape: {y.shape}")
print(f"X dtype: {X.dtype}, value range: [{X.min()}, {X.max()}]")
print(f"First encoded sequence: {X[0]}")


X shape: (575061, 40)  (sessions, MAX_SEQ_LEN)
y shape: (575061,)
X dtype: int32, value range: [0, 29]
First encoded sequence: [25 15 25 25  3  3 29 29  3 29 19 19 19 26 25  8 26 25 10 18 19 19 23 18
 26 26 25 25  8 10 19 19 25 26 25  8 23 23 23 23]


In [33]:

# Cell 11 - Stratified train/val/test split
# Stratify on the label so the 2.9% anomaly rate is preserved in every split.

from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE, stratify=y_temp, random_state=SEED
)

for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    n = len(yy)
    a = int(yy.sum())
    print(f"{name}: {n} sessions, {a} anomalies ({100*a/n:.2f}%)")


train: 415480 sessions, 12165 anomalies (2.93%)
val: 73321 sessions, 2147 anomalies (2.93%)
test: 86260 sessions, 2526 anomalies (2.93%)


In [34]:

import json

np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUT_DIR, "y_test.npy"), y_test)

metadata = {
    "dataset": "HDFS_v1",
    "source_url": HDFS_ZIP_URL,
    "task": "binary sequence anomaly detection (Normal=0 / Anomaly=1)",
    "role": "LogSentinel — auxiliary detector feeding structured scores to the LLM orchestrator",
    "max_seq_len": MAX_SEQ_LEN,
    "pad_value": PAD_VALUE,
    "unknown_id": UNKNOWN_ID,
    "vocab_size": VOCAB_SIZE,
    "seed": SEED,
    "counts": {
        "total": int(n_total),
        "train": int(len(y_train)),
        "val": int(len(y_val)),
        "test": int(len(y_test)),
    },
    "anomaly_rate_overall": float(n_anom / n_total),
    "vocab": vocab,
}

with open(os.path.join(OUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved to local disk:")
for fn in os.listdir(OUT_DIR):
    size = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6
    print(f"  {fn}  ({size:.2f} MB)")


Saved to local disk:
  y_train.npy  (1.66 MB)
  X_val.npy  (11.73 MB)
  metadata.json  (0.00 MB)
  X_test.npy  (13.80 MB)
  X_train.npy  (66.48 MB)
  y_val.npy  (0.29 MB)
  y_test.npy  (0.35 MB)


In [35]:

import shutil

os.makedirs(DRIVE_OUT, exist_ok=True)
for fn in os.listdir(OUT_DIR):
    shutil.copy(os.path.join(OUT_DIR, fn), os.path.join(DRIVE_OUT, fn))

print(f"Copied all tensors to Drive: {DRIVE_OUT}")
for fn in os.listdir(DRIVE_OUT):
    print(f"  {fn}")


Copied all tensors to Drive: /content/drive/MyDrive/Aegis-CyberSec-Guard/logsentinel/dataV1
  y_train.npy
  X_val.npy
  metadata.json
  X_test.npy
  X_train.npy
  y_val.npy
  y_test.npy


In [36]:

# Cell 14 - Reload verification
# Read the tensors back exactly as the training script will. Fail here (cheap), not there.

Xtr = np.load(os.path.join(DRIVE_OUT, "X_train.npy"))
ytr = np.load(os.path.join(DRIVE_OUT, "y_train.npy"))
with open(os.path.join(DRIVE_OUT, "metadata.json")) as f:
    meta = json.load(f)

assert Xtr.shape[1] == MAX_SEQ_LEN, "sequence length mismatch"
assert Xtr.shape[0] == ytr.shape[0], "X/y length mismatch"
assert set(np.unique(ytr)).issubset({0, 1}), "labels must be binary"

print("Reload verification passed.")
print(f"  X_train: {Xtr.shape}, vocab_size: {meta['vocab_size']}")
print(f"  anomaly rate in train: {100*ytr.mean():.2f}%")
print("\nETL complete. Tensors ready for the CNN1D training script.")


Reload verification passed.
  X_train: (415480, 40), vocab_size: 31
  anomaly rate in train: 2.93%

ETL complete. Tensors ready for the CNN1D training script.
